In [2]:
import ee

from repository_ee import (
    Exports,
    GeometryUtils,
    Landsat8,
    MapBiomasLabels,
    MapLayers,
    PeriodoAgricola,
    TimeSeries,
    initialize_earth_engine,
)

In [30]:
# 1) Inicialização e área de estudo da fonte (retângulo no Tocantins).
initialize_earth_engine()

COORDS_TOCANTINS = [
    [-48.46995289774661, -10.859497023647087],
    [-47.84922535868411, -10.859497023647087],
    [-47.84922535868411, -10.362094203375317],
    [-48.46995289774661, -10.362094203375317],
    [-48.46995289774661, -10.859497023647087],
]
aoi = GeometryUtils.polygon(COORDS_TOCANTINS)
print(GeometryUtils.metrics(aoi))



{'area_m2': 3752254309.897163, 'area_hectares': 375225.4309897163, 'area_km2': 3752.254309897163, 'perimetro_m': 245885.64120605137, 'perimetro_km': 245.88564120605136}


In [31]:
# 2) Coleção Landsat e produtos derivados.
periodo = PeriodoAgricola.tocantins(2025)
landsat = Landsat8.collection(aoi, periodo)
feature_space = Landsat8.feature_space(landsat, aoi, periodo)
pico_verde = Landsat8.green_peak(landsat, aoi, periodo)
classes = MapBiomasLabels.pasture_and_agriculture(periodo.ano, aoi)
imagem_completa = feature_space.addBands(classes)

In [ ]:
# 3) Mapa com camadas especializadas. No Jupyter, deixe ``mapa`` na última linha.
mapa = MapLayers.create(aoi, zoom=8)
MapLayers.add_aoi(mapa, aoi)
MapLayers.add_rgb(mapa, feature_space, "RGB — mediana anual")
MapLayers.add_ndvi(mapa, pico_verde, "NDVI — pico vegetativo")
MapLayers.add_classes(mapa, imagem_completa)
mapa

Map(center=[-10.61088033468291, -48.15958912821537], controls=(WidgetControl(options=['position', 'transparent…

In [33]:
# 4) Exportações independentes: escolha quais tarefas quer criar e iniciar.
# Cada exportação respeita EXATAMENTE a AOI de COORDS_TOCANTINS.
task_rgb = Exports.rgb_to_drive(feature_space, aoi, "tocantins_2025_rgb")
task_ndvi = Exports.ndvi_to_drive(pico_verde, aoi, "tocantins_2025_ndvi_pico")
task_feature_space = Exports.feature_space_to_drive(feature_space, aoi, "tocantins_2025_feature_space")

# Descomente somente os produtos que deseja enviar para o Drive.
# task_rgb.start()
# task_ndvi.start()
# task_feature_space.start()